<a href="https://colab.research.google.com/github/lorenzochesta/Project_work_Computer_Vision/blob/main/Segmentazione_vasi_retinici.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import os
import cv2
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import albumentations as A
from scipy.ndimage import distance_transform_edt

In [ ]:
print("PyTorch:",torch.__version__)
print("CUDA disponibile:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:",torch.cuda.get_device_name(0))
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

PyTorch: 2.11.0+cu128
CUDA disponibile: True
GPU: Tesla T4


## Versione 1

Data Augmentation sulle immagini RGB. Conversione delle immagini RGB in scala di grigio. Estrazione delle patch ed addestramento di U-Net.

### Definizione del dataset

In [ ]:
# creo la classe RetinaPatchDataset che eredita da Dataset
class RetinaPatchDataset(Dataset):
    def __init__(self, immagini_dir, maschere_dir, patch_size=96, patches_per_image=10, transform=None, is_train=True):
        # is_train = True --> estrae patch in posizioni casuali
        # is_train = False --> estrae patch in posizioni fisse (riproducibili)

        self.immagini_dir = immagini_dir
        self.maschere_dir = maschere_dir
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.transform = transform
        self.is_train = is_train

        self.lista_immagini = sorted(os.listdir(immagini_dir))
        self.lista_maschere = sorted(os.listdir(maschere_dir))

        assert len(self.lista_immagini) == len(self.lista_maschere), "Il numero di immagini e maschere non coincide."

    def __len__(self):
        return len(self.lista_immagini) * self.patches_per_image

    def __getitem__(self, idx):
        img_idx = idx // self.patches_per_image
        patch_idx = idx % self.patches_per_image

        img_path = os.path.join(self.immagini_dir, self.lista_immagini[img_idx])
        mask_path = os.path.join(self.maschere_dir, self.lista_maschere[img_idx])

        immagine = cv2.imread(img_path)
        immagine = cv2.cvtColor(immagine, cv2.COLOR_BGR2RGB).astype(np.float32)
        maschera = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE).astype(np.float32)

        # data augmentation
        if self.transform:
            augmented = self.transform(image=immagine, mask=maschera)
            immagine = augmented['image']
            maschera = augmented['mask']

        # conversione in scala di grigi
        grigio = 0.299 * immagine[:,:,0] + 0.587 * immagine[:,:,1] + 0.114 * immagine[:,:,2]

        # standardizzazione Z-score
        media = np.mean(grigio)
        dev_std = np.std(grigio)
        if dev_std == 0: dev_std = 1e-6
        grigio_standardizzato = (grigio - media) / dev_std

        # normalizzazione Min-Max in [0.0, 1.0]
        min_val = np.min(grigio_standardizzato)
        max_val = np.max(grigio_standardizzato)
        if (max_val - min_val) == 0: max_val += 1e-6
        grigio_normalizzato = (grigio_standardizzato - min_val) / (max_val - min_val)

        # normalizzazione Maschera
        maschera_binaria = ((maschera / 255.0) > 0.5).astype(np.float32)

        # estrazione Patch
        H, W = grigio_normalizzato.shape
        max_h = H - self.patch_size
        max_w = W - self.patch_size

        if self.is_train:
            start_h = np.random.randint(0, max_h)
            start_w = np.random.randint(0, max_w)
        else:
            rng = np.random.default_rng(patch_idx)
            start_h = rng.integers(0, max_h)
            start_w = rng.integers(0, max_w)

        patch_img = grigio_normalizzato[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]
        patch_mask = maschera_binaria[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]

        # conversione in tensori
        patch_img_t = torch.tensor(patch_img, dtype=torch.float32).unsqueeze(0) # Forma: (1, P, P)

        patch_mask_t = torch.tensor(patch_mask, dtype=torch.long) # Forma: (P, P)

        return patch_img_t, patch_mask_t

In [ ]:
# definisco le operazioni di data augmentation da applicare al training set (sia all'immagine che alla maschera)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2)
])

test_transform=None

In [ ]:
# inizializzo i 3 dataset

# DRIVE --> training
dataset_train = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/drive/training/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/drive/training/ground_truth',
    patch_size=96,
    patches_per_image=15,
    transform=train_transform,
    is_train=True
)

# STARE --> test
dataset_test = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/stare/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/stare/ground_truth_ah',
    patch_size=96,
    patches_per_image=10,
    transform=test_transform,
    is_train=False
)

# CHASE --> generalizzazione
dataset_generalizzazione = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/chase/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/chase/ground_truth_ann1',
    patch_size=96,
    patches_per_image=10,
    transform=test_transform,
    is_train=False
)


# DataLoader
train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False, num_workers=2)
gen_loader = DataLoader(dataset_generalizzazione, batch_size=8, shuffle=False, num_workers=2)

### Definizione del modello

In [ ]:
class DoubleConv(nn.Module):
    """Blocco di due convoluzioni consecutive"""

    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.2), # Dropout tra due convoluzioni consecutive
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class RetinalUNet(nn.Module):
    def __init__(self):
        super(RetinalUNet, self).__init__()

        # Encoder
        self.enc1 = DoubleConv(1, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(64, 128)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(64, 32)

        # Output (2 classi)
        self.final_conv = nn.Conv2d(32, 2, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        p1 = self.pool1(x1)

        x2 = self.enc2(p1)
        p2 = self.pool2(x2)

        b = self.bottleneck(p2)

        # Decoder
        up_b = self.up1(b)
        merge1 = torch.cat([x2, up_b], dim=1)
        d1 = self.dec1(merge1)

        up_d1 = self.up2(d1)
        merge2 = torch.cat([x1, up_d1], dim=1)
        d2 = self.dec2(merge2)

        # Output
        out = self.final_conv(d2)

        return F.log_softmax(out, dim=1)

### Definizione del ciclo di addestramento

In [ ]:
class DiceFocalLoss(nn.Module):
    def __init__(self, weight_dice=1.0, weight_focal=1.0, focal_gamma=2.0, focal_alpha=[0.2, 0.8]):
        super(DiceFocalLoss, self).__init__()
        self.weight_dice = weight_dice
        self.weight_focal = weight_focal
        self.focal_gamma = focal_gamma
        self.register_buffer('focal_alpha', torch.tensor(focal_alpha, dtype=torch.float32))

    def forward(self, predizioni, target):
        if target.dim() == 4 and target.shape[1] == 1:
            target_labels = target.squeeze(1).long()
        elif target.dim() == 3:
            target_labels = target.long()
        else:
            target_labels = target.long()

        target_one_hot = F.one_hot(target_labels, num_classes=2).permute(0, 3, 1, 2).float()

        # calcolo la dice loss
        prob = torch.exp(predizioni)

        intersezione = torch.sum(prob * target_one_hot, dim=(2, 3))
        unione = torch.sum(prob, dim=(2, 3)) + torch.sum(target_one_hot, dim=(2, 3))
        dice = (2. * intersezione + 1e-5) / (unione + 1e-5)
        dice_loss = 1.0 - torch.mean(dice)

        # calcolo la focal loss
        ce_loss = F.nll_loss(predizioni, target_labels, reduction='none')
        pt = torch.exp(-ce_loss)

        alpha_pixel = self.focal_alpha[target_labels]

        focal_loss = alpha_pixel * ((1 - pt) ** self.focal_gamma) * ce_loss
        focal_loss = focal_loss.mean()

        # combino dice e focal loss
        total_loss = (self.weight_dice * dice_loss) + (self.weight_focal * focal_loss)
        return total_loss

In [ ]:
def calcola_metriche(pred_probs, true_masks):
    pred_binarie = (pred_probs > 0.6).astype(int)

    acc = accuracy_score(true_masks, pred_binarie)
    prec = precision_score(true_masks, pred_binarie, zero_division=0)
    rec = recall_score(true_masks, pred_binarie, zero_division=0)
    f1 = f1_score(true_masks, pred_binarie, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(true_masks, pred_binarie, labels=[0, 1]).ravel()
    spec = tn / (tn + fp + 1e-6)

    try:
        auc = roc_auc_score(true_masks, pred_probs)
    except ValueError:
        auc = 0.5

    return {
        "accuracy": acc, "precision": prec, "recall": rec,
        "specificity": spec, "f1_score": f1, "auc": auc
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modello = RetinalUNet().to(device)

criterion = DiceFocalLoss(
    weight_dice=1.0,
    weight_focal=2.0,
    focal_gamma=2.0,
    focal_alpha=[0.15,0.85]
).to(device)

optimizer = optim.Adam(
    modello.parameters(),
    lr=3e-4
)


num_epoche = 50

print("\nMetriche di Training (DRIVE):\n")

for epoca in range(num_epoche):
    print(f"Epoca {epoca+1}/{num_epoche}:")

    modello.train()
    loss_totale_train = 0

    tutti_i_target_train = []
    tutte_le_probabilita_train = []

    for immagini, maschere in train_loader:
        immagini = immagini.to(device)
        maschere = maschere.to(device)

        predizioni = modello(immagini)
        loss = criterion(predizioni, maschere)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_totale_train += loss.item()

        probabilities = torch.exp(predizioni)
        prob_foreground = probabilities[:, 1, :, :]

        tutti_i_target_train.append(maschere.cpu().numpy().flatten())
        tutte_le_probabilita_train.append(prob_foreground.cpu().detach().numpy().flatten())

    tutti_i_target_train = np.concatenate(tutti_i_target_train)
    tutte_le_probabilita_train = np.concatenate(tutte_le_probabilita_train)

    metriche_train = calcola_metriche(tutte_le_probabilita_train, tutti_i_target_train)
    loss_media_epoch = loss_totale_train / len(train_loader)

    print(f"Loss: {loss_media_epoch:.4f}"
           f" Accuracy: {metriche_train['accuracy']:.4f}"
           f" Precision: {metriche_train['precision']:.4f}"
           f" Recall: {metriche_train['recall']:.4f}"
           f" Specificity: {metriche_train['specificity']:.4f}"
           f" F1-Score: {metriche_train['f1_score']:.4f}"
           f" AUC-ROC: {metriche_train['auc']:.4f}\n"
    )



Metriche di Training (DRIVE):

Epoca 1/50:
Loss: 0.6460 Accuracy: 0.8791 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.4809

Epoca 2/50:
Loss: 0.6433 Accuracy: 0.8864 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.4998

Epoca 3/50:
Loss: 0.6434 Accuracy: 0.8851 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.5627

Epoca 4/50:
Loss: 0.6380 Accuracy: 0.8788 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.5396

Epoca 5/50:
Loss: 0.6237 Accuracy: 0.8838 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.5672

Epoca 6/50:
Loss: 0.6266 Accuracy: 0.8840 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.5688

Epoca 7/50:
Loss: 0.6200 Accuracy: 0.8852 Precision: 0.0000 Recall: 0.0000 Specificity: 1.0000 F1-Score: 0.0000 AUC-ROC: 0.5878

Epoca 8/50:
Loss: 0.6220 Accuracy: 0.8851 Precision: 0.0000 Recal

## Versione 2 (migliore)

Dei 3 canali RGB tengo solo il verde, perchè offre il miglior contrasto vaso/sfondo (il rosso e il blu invece introducono informazioni meno rilevanti e tendono a ridurre il contrasto).
Metto in risalto il verde tramite CLAHE. Definisco una _soglia_vasi_ per scartare da DRIVE le immagini in cui il numero di pixel contenenti un vaso è minore di _soglia_vasi_.

### Definizione del dataset

In [ ]:
class RetinaPatchDataset(Dataset):
    def __init__(self, immagini_dir, maschere_dir, patch_size=96, patches_per_image=10, transform=None, is_train=True, soglia_vasi=0.01):

        self.immagini_dir = immagini_dir
        self.maschere_dir = maschere_dir
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.transform = transform
        self.is_train = is_train
        self.soglia_vasi = soglia_vasi # nuova variabile per la soglia

        self.lista_immagini = sorted(os.listdir(immagini_dir))
        self.lista_maschere = sorted(os.listdir(maschere_dir))

        assert len(self.lista_immagini) == len(self.lista_maschere), "Il numero di immagini e maschere non coincide."

    def __len__(self):
        return len(self.lista_immagini) * self.patches_per_image

    def __getitem__(self, idx):
        img_idx = idx // self.patches_per_image
        patch_idx = idx % self.patches_per_image

        img_path = os.path.join(self.immagini_dir, self.lista_immagini[img_idx])
        mask_path = os.path.join(self.maschere_dir, self.lista_maschere[img_idx])

        immagine = cv2.imread(img_path)
        immagine = cv2.cvtColor(immagine, cv2.COLOR_BGR2RGB)
        maschera = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # prendo solo il verde
        canale_verde = immagine[:, :, 1]

        # CLAHE
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        verde_equalizzato = clahe.apply(canale_verde).astype(np.float32)

        # Standardizzazione e Normalizzazione Min-Max
        media = np.mean(verde_equalizzato)
        dev_std = np.std(verde_equalizzato) if np.std(verde_equalizzato) > 0 else 1e-6
        grigio_standardizzato = (verde_equalizzato - media) / dev_std

        min_val = np.min(grigio_standardizzato)
        max_val = np.max(grigio_standardizzato)
        diff = (max_val - min_val) if (max_val - min_val) > 0 else 1e-6
        grigio_normalizzato = (grigio_standardizzato - min_val) / diff

        maschera_binaria = (maschera > 0).astype(np.float32)

        H, W = grigio_normalizzato.shape
        max_h = H - self.patch_size
        max_w = W - self.patch_size

        if self.is_train:
            tentativi = 0
            while tentativi < 50:
                start_h = np.random.randint(0, max_h)
                start_w = np.random.randint(0, max_w)

                patch_mask = maschera_binaria[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]
                percentuale_vasi = np.sum(patch_mask > 0) / (self.patch_size * self.patch_size)

                if percentuale_vasi >= self.soglia_vasi:
                    break
                tentativi += 1

            patch_img = grigio_normalizzato[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]
        else:
            rng = np.random.default_rng(patch_idx)
            start_h = rng.integers(0, max_h)
            start_w = rng.integers(0, max_w)

            patch_img = grigio_normalizzato[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]
            patch_mask = maschera_binaria[start_h:start_h+self.patch_size, start_w:start_w+self.patch_size]

        # data augmentation sulle patch
        if self.transform and self.is_train:
            augmented = self.transform(image=patch_img, mask=patch_mask)
            patch_img = augmented['image']
            patch_mask = augmented['mask']

        patch_img_t = torch.tensor(patch_img, dtype=torch.float32).unsqueeze(0)
        patch_mask_t = torch.tensor(patch_mask, dtype=torch.long)

        return patch_img_t, patch_mask_t

In [ ]:
# definisco le operazioni di data augmentation da applicare al training set (sia all'immagine che alla maschera)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2)
])

test_transform = None

In [ ]:
# inizializzazione dei 3 dataset

# DRIVE --> training
dataset_train = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/drive/training/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/drive/training/ground_truth',
    patch_size=96,
    patches_per_image=15,
    transform=train_transform,
    is_train=True,
    soglia_vasi=0.01 # soglia per scartare le immagini con pochi vasi
)

# STARE --> test
dataset_test = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/stare/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/stare/ground_truth_ah',
    patch_size=96,
    patches_per_image=10,
    transform=test_transform,
    is_train=False # patch con posizione fissa
)

# CHASE --> generalizzazione
dataset_generalizzazione = RetinaPatchDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/chase/images',
    maschere_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/chase/ground_truth_ann1',
    patch_size=96,
    patches_per_image=10,
    transform=test_transform,
    is_train=False
)


# DataLoader
train_loader = DataLoader(dataset_train, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset_test, batch_size=32, shuffle=False, num_workers=2)
gen_loader = DataLoader(dataset_generalizzazione, batch_size=8, shuffle=False, num_workers=2)

### Definizione del modello

In [ ]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Dropout2d(p=0.2),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class RetinalUNet(nn.Module):
    def __init__(self):
        super(RetinalUNet, self).__init__()

        # Encoder
        self.enc1 = DoubleConv(1, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(64, 128)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(64, 32)

        # Output
        self.final_conv = nn.Conv2d(32, 2, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        p1 = self.pool1(x1)

        x2 = self.enc2(p1)
        p2 = self.pool2(x2)

        b = self.bottleneck(p2)

        # Decoder
        up_b = self.up1(b)
        merge1 = torch.cat([x2, up_b], dim=1)
        d1 = self.dec1(merge1)

        up_d1 = self.up2(d1)
        merge2 = torch.cat([x1, up_d1], dim=1)
        d2 = self.dec2(merge2)

        # Output
        out = self.final_conv(d2)

        return F.log_softmax(out, dim=1)

### Definizione del ciclo di addestramento

In [ ]:
class DiceFocalLoss(nn.Module):
    def __init__(self, weight_dice=1.0, weight_focal=1.0, focal_gamma=2.0, focal_alpha=[0.2, 0.8]):
        super(DiceFocalLoss, self).__init__()
        self.weight_dice = weight_dice
        self.weight_focal = weight_focal
        self.focal_gamma = focal_gamma
        self.register_buffer('focal_alpha', torch.tensor(focal_alpha, dtype=torch.float32))

    def forward(self, predizioni, target):

        if target.dim() == 4 and target.shape[1] == 1:
            target_labels = target.squeeze(1).long()
        elif target.dim() == 3:
            target_labels = target.long()
        else:
            target_labels = target.long()


        target_one_hot = F.one_hot(target_labels, num_classes=2).permute(0, 3, 1, 2).float()

        # Calcolo Dice Loss
        prob = torch.exp(predizioni)

        intersezione = torch.sum(prob * target_one_hot, dim=(2, 3))
        unione = torch.sum(prob, dim=(2, 3)) + torch.sum(target_one_hot, dim=(2, 3))
        dice = (2. * intersezione + 1e-5) / (unione + 1e-5)
        dice_loss = 1.0 - torch.mean(dice)

        # Calcolo Focal Loss
        ce_loss = F.nll_loss(predizioni, target_labels, reduction='none')
        pt = torch.exp(-ce_loss)

        alpha_pixel = self.focal_alpha[target_labels]

        focal_loss = alpha_pixel * ((1 - pt) ** self.focal_gamma) * ce_loss
        focal_loss = focal_loss.mean()

        # Combino Dice e Focal Loss
        total_loss = (self.weight_dice * dice_loss) + (self.weight_focal * focal_loss)
        return total_loss

In [ ]:
def calcola_metriche(pred_probs, true_masks):

    pred_binarie = (pred_probs > 0.6).astype(int)

    acc = accuracy_score(true_masks, pred_binarie)
    prec = precision_score(true_masks, pred_binarie, zero_division=0)
    rec = recall_score(true_masks, pred_binarie, zero_division=0)
    f1 = f1_score(true_masks, pred_binarie, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(true_masks, pred_binarie, labels=[0, 1]).ravel()
    spec = tn / (tn + fp + 1e-6)

    try:
        auc = roc_auc_score(true_masks, pred_probs)
    except ValueError:
        # Gestisce il caso raro in cui nel batch ci siano solo pixel di sfondo (niente vasi)
        auc = 0.5

    return {
        "accuracy": acc, "precision": prec, "recall": rec,
        "specificity": spec, "f1_score": f1, "auc": auc
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modello = RetinalUNet().to(device)

criterion = DiceFocalLoss(
    weight_dice=1.0,
    weight_focal=2.0,
    focal_gamma=2.0,
    focal_alpha=[0.15,0.85]
).to(device)

optimizer = optim.Adam(
    modello.parameters(),
    lr=3e-4
)

# Training su DRIVE
num_epoche = 50

if 'epoca_partenza' not in locals():
    epoca_partenza = 0

if 'storico_metriche' not in locals():
    storico_metriche = {
        'train_loss': [], 'train_acc': [], 'train_prec': [],
        'train_rec': [], 'train_spec': [], 'train_f1': [], 'train_auc': []
    }


checkpoint_dir = "/content/drive/MyDrive/ProjectWork_CV/retina_checkpoints"
checkpoint_path = os.path.join(checkpoint_dir, "checkpoint_corrente.pth")


os.makedirs(checkpoint_dir, exist_ok=True)

print("\nMetriche di Training (DRIVE):\n")

for epoca in range(epoca_partenza, num_epoche):
    print(f"Epoca {epoca+1}/{num_epoche}:")

    modello.train()
    loss_totale_train = 0

    tutti_i_target_train = []
    tutte_le_probabilita_train = []

    for immagini, maschere in train_loader:
        immagini = immagini.to(device)
        maschere = maschere.to(device)

        # Forward pass e calcolo della Loss
        predizioni = modello(immagini)
        loss = criterion(predizioni, maschere)

        # Backward pass e ottimizzazione
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_totale_train += loss.item()

        with torch.no_grad():
            # estraggo le probabilità del canale dei vasi (canale 1)
            prob_train = torch.exp(predizioni[:, 1, :, :])
            tutte_le_probabilita_train.append(prob_train.cpu().numpy().ravel())

            maschera_binaria = (maschere > 0).cpu().numpy().astype(int).ravel()
            tutti_i_target_train.append(maschera_binaria)

    tutte_le_probabilita_train = np.concatenate(tutte_le_probabilita_train)
    tutti_i_target_train = np.concatenate(tutti_i_target_train)

    metriche_train = calcola_metriche(tutte_le_probabilita_train, tutti_i_target_train)
    loss_media_epoch = loss_totale_train / len(train_loader)

    print(f"Loss: {loss_media_epoch:.4f}"
          f" Accuracy: {metriche_train['accuracy']:.4f}"
          f" Precision: {metriche_train['precision']:.4f}"
          f" Recall: {metriche_train['recall']:.4f}"
          f" Specificity: {metriche_train['specificity']:.4f}"
          f" F1-Score: {metriche_train['f1_score']:.4f}"
          f" AUC-ROC: {metriche_train['auc']:.4f}\n"
    )

    # aggiorno lo storico delle metriche
    storico_metriche['train_loss'].append(loss_media_epoch)
    storico_metriche['train_acc'].append(metriche_train['accuracy'])
    storico_metriche['train_prec'].append(metriche_train['precision'])
    storico_metriche['train_rec'].append(metriche_train['recall'])
    storico_metriche['train_spec'].append(metriche_train['specificity'])
    storico_metriche['train_f1'].append(metriche_train['f1_score'])
    storico_metriche['train_auc'].append(metriche_train['auc'])

    checkpoint = {
        'epoca': epoca + 1,
        'model_state_dict': modello.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_media_epoch,
        'storico_metriche': storico_metriche
    }
    torch.save(checkpoint, checkpoint_path)


Metriche di Training (DRIVE):

Epoca 1/50:
Loss: 0.6863 Accuracy: 0.7268 Precision: 0.2249 Recall: 0.5859 Specificity: 0.7447 F1-Score: 0.3250 AUC-ROC: 0.7196

Epoca 2/50:
Loss: 0.5861 Accuracy: 0.8700 Precision: 0.4882 Recall: 0.7125 Specificity: 0.8926 F1-Score: 0.5794 AUC-ROC: 0.8726

Epoca 3/50:
Loss: 0.5504 Accuracy: 0.8978 Precision: 0.5403 Recall: 0.7327 Specificity: 0.9192 F1-Score: 0.6219 AUC-ROC: 0.8971

Epoca 4/50:
Loss: 0.5238 Accuracy: 0.9103 Precision: 0.5851 Recall: 0.7304 Specificity: 0.9335 F1-Score: 0.6497 AUC-ROC: 0.9063

Epoca 5/50:
Loss: 0.4996 Accuracy: 0.9182 Precision: 0.6310 Recall: 0.7225 Specificity: 0.9441 F1-Score: 0.6736 AUC-ROC: 0.9159

Epoca 6/50:
Loss: 0.4889 Accuracy: 0.9224 Precision: 0.6488 Recall: 0.7216 Specificity: 0.9487 F1-Score: 0.6833 AUC-ROC: 0.9163

Epoca 7/50:
Loss: 0.4714 Accuracy: 0.9239 Precision: 0.6504 Recall: 0.7362 Specificity: 0.9484 F1-Score: 0.6907 AUC-ROC: 0.9264

Epoca 8/50:
Loss: 0.4522 Accuracy: 0.9275 Precision: 0.6869 Recal

In [ ]:
# Ripristino
modello = RetinalUNet().to(device)

criterion = DiceFocalLoss(
    weight_dice=1.0,
    weight_focal=2.0,
    focal_gamma=2.0,
    focal_alpha=[0.15,0.85]
).to(device)

optimizer = optim.Adam(
    modello.parameters(),
    lr=3e-4
)


storico_metriche = {
    'train_loss': [], 'train_acc': [], 'train_prec': [], 'train_rec': [], 'train_spec': [], 'train_f1': [], 'train_auc': []
}


checkpoint_dir = "/content/drive/MyDrive/ProjectWork_CV/retina_checkpoints"
checkpoint_path = os.path.join(checkpoint_dir, "checkpoint_corrente.pth")


if os.path.exists(checkpoint_path):
    print("Trovato checkpoint precedente! Ripristino dello stato...")

    # Carica il dizionario nella memoria
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Ripristina i pesi della rete e lo stato di Adam
    modello.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Ripristina lo storico delle metriche salvate fino a quel momento
    storico_metriche = checkpoint['storico_metriche']

    # Leggi l'epoca da cui ripartire
    epoca_partenza = checkpoint['epoca']
    print(f"Stato ripristinato. L'addestramento riprenderà dall'epoca {epoca_partenza + 1}")
else:
    print("Nessun checkpoint trovato. L'addestramento partirà da zero (Epoca 1).")
    epoca_partenza = 0

Trovato checkpoint precedente! Ripristino dello stato...
Stato ripristinato. L'addestramento riprenderà dall'epoca 51


In [ ]:
# Test su STARE
modello.eval()
loss_totale_test = 0

tutti_i_target_test = []
tutte_le_probabilita_test = []

print("\nMetriche di Test (STARE):\n")

with torch.no_grad():
    for immagini_test, maschere_test in test_loader:
        immagini_test = immagini_test.to(device)
        maschere_test = maschere_test.to(device)

        # Forward pass
        predizioni_test = modello(immagini_test)

        # Calcolo della loss sulle patch
        loss_test = criterion(predizioni_test, maschere_test)
        loss_totale_test += loss_test.item()

        # Estraggo le probabilità del canale 1 (vasi)
        prob_test = torch.exp(predizioni_test[:, 1, :, :])
        tutte_le_probabilita_test.append(prob_test.cpu().numpy().ravel())

        maschera_test_binaria = (maschere_test > 0).cpu().numpy().astype(int).ravel()
        tutti_i_target_test.append(maschera_test_binaria)


tutte_le_probabilita_test = np.concatenate(tutte_le_probabilita_test)
tutti_i_target_test = np.concatenate(tutti_i_target_test)
metriche_test = calcola_metriche(tutte_le_probabilita_test, tutti_i_target_test)

print(f"Loss globale: {loss_totale_test/len(test_loader):.4f}")
print(f"Accuracy:     {metriche_test['accuracy']:.4f}")
print(f"Precision:    {metriche_test['precision']:.4f}")
print(f"Recall:       {metriche_test['recall']:.4f}")
print(f"Specificity:  {metriche_test['specificity']:.4f}")
print(f"F1-Score:     {metriche_test['f1_score']:.4f}")
print(f"AUC-ROC:      {metriche_test['auc']:.4f}")


Metriche di Test (STARE):

Loss globale: 0.3449
Accuracy:     0.9415
Precision:    0.6404
Recall:       0.8348
Specificity:  0.9523
F1-Score:     0.7248
AUC-ROC:      0.9610


In [ ]:
# Valuto la capacità di generalizzazione della rete su CHASE
modello.eval()
tutti_i_target_gen = []
tutte_le_probabilita_gen = []

print("\nGeneralizzazione (CHASE):\n")

with torch.no_grad():
    for immagini_gen, maschere_gen in gen_loader:
        immagini_gen = immagini_gen.to(device)

        # Forward pass
        predizioni_gen = modello(immagini_gen)

        # Estraggo le probabilità del canale 1 (vasi)
        prob_gen = torch.exp(predizioni_gen[:, 1, :, :])

        tutte_le_probabilita_gen.append(prob_gen.cpu().numpy().ravel())

        maschera_gen_binaria = (maschere_gen > 0).cpu().numpy().astype(int).ravel()
        tutti_i_target_gen.append(maschera_gen_binaria)

tutte_le_probabilita_gen = np.concatenate(tutte_le_probabilita_gen)
tutti_i_target_gen = np.concatenate(tutti_i_target_gen)
metriche_gen = calcola_metriche(tutte_le_probabilita_gen, tutti_i_target_gen)


print(f"Accuracy:    {metriche_gen['accuracy']:.4f}")
print(f"Precision:   {metriche_gen['precision']:.4f}")
print(f"Recall:      {metriche_gen['recall']:.4f}")
print(f"Specificity: {metriche_gen['specificity']:.4f}")
print(f"F1-Score:    {metriche_gen['f1_score']:.4f}")
print(f"AUC-ROC:     {metriche_gen['auc']:.4f}")


Generalizzazione (CHASE):

Accuracy:    0.9106
Precision:   0.6672
Recall:      0.5196
Specificity: 0.9643
F1-Score:    0.5842
AUC-ROC:     0.8746


## Metriche

Calcolate su STARE.

### Tortuosità

In [ ]:
from skimage.morphology import skeletonize
from scipy.ndimage import label

In [ ]:
def calcola_tortuosita(maschera_predetta, min_pixel_lunghezza=15):
    """
    Calcola la tortuosità media dei vasi come lunghezza dell'arco / lunghezza della corda.
    La lunghezza dell'arco viene stimata come il numero di pixel che compongono lo scheletro.
    La corda è identificata dai due pixel che hanno distanza euclidea massima tra loro.
    """
    binary_mask = (maschera_predetta > 0).astype(np.uint8)
    scheletro = skeletonize(binary_mask).astype(np.uint8)

    # Identificazione e rimozione dei bivi
    kernel = np.array([[1, 1, 1],
                       [1, 0, 1],
                       [1, 1, 1]], dtype=np.uint8)
    vicini = cv2.filter2D(scheletro, -1, kernel) * scheletro
    punti_bivio = (vicini > 2).astype(np.uint8)

    # Spezzo lo scheletro
    scheletro_segmentato = (scheletro - punti_bivio) > 0
    scheletro_segmentato = scheletro_segmentato.astype(np.uint8)

    segmenti_etichettati, num_features = label(scheletro_segmentato)

    tortuosita_totale = 0
    segmenti_validi = 0

    for i in range(1, num_features + 1):
        punti = np.argwhere(segmenti_etichettati == i)

        if len(punti) < min_pixel_lunghezza:
            continue

        # Arco come numero di pixel del segmento
        lunghezza_arco = len(punti)

        # Corda come massima distanza tra le coppie di punti del segmento
        diff = punti[:, np.newaxis, :] - punti[np.newaxis, :, :]
        distanze_al_quadrato = np.sum(diff ** 2, axis=-1)
        lunghezza_corda = np.sqrt(np.max(distanze_al_quadrato))

        if lunghezza_corda == 0:
            continue

        tortuosita = lunghezza_arco / lunghezza_corda

        if tortuosita >= 1.0:
            tortuosita_totale += tortuosita
            segmenti_validi += 1

    return tortuosita_totale / segmenti_validi if segmenti_validi > 0 else 1.0

In [ ]:
# Estendo il codice del test su STARE scritto in precedenza includendo anche la Tortuosità
modello.eval()
loss_totale_test = 0
tortuosita_totale_test = 0 # accumulatore per la tortuosità
conteggio_patch_tortuosita = 0

tutti_i_target_test = []
tutte_le_probabilita_test = []

print("\nMetriche di Test (STARE):\n")

with torch.no_grad():
    for immagini_test, maschere_test in test_loader: # carica le patch fisse di STARE
        immagini_test = immagini_test.to(device)
        maschere_test = maschere_test.to(device)

        predizioni_test = modello(immagini_test)

        loss_test = criterion(predizioni_test, maschere_test)
        loss_totale_test += loss_test.item()

        prob_test = torch.exp(predizioni_test[:, 1, :, :])
        tutte_le_probabilita_test.append(prob_test.cpu().numpy().ravel())

        maschera_test_binaria = (maschere_test > 0).cpu().numpy().astype(int).ravel()
        tutti_i_target_test.append(maschera_test_binaria)

        # Calcolo della tortuosità sulle predizioni del batch
        predizioni_binarie_np = (prob_test > 0.5).cpu().numpy().astype(np.uint8)

        # Itero su ogni patch nel batch corrente
        for b in range(predizioni_binarie_np.shape[0]):
            tortuosita_patch = calcola_tortuosita(predizioni_binarie_np[b])
            tortuosita_totale_test += tortuosita_patch
            conteggio_patch_tortuosita += 1

tutte_le_probabilita_test = np.concatenate(tutte_le_probabilita_test)
tutti_i_target_test = np.concatenate(tutti_i_target_test)
metriche_test = calcola_metriche(tutte_le_probabilita_test, tutti_i_target_test)

# Calcolo della tortuosità media globale
tortuosita_media_global = tortuosita_totale_test / conteggio_patch_tortuosita if conteggio_patch_tortuosita > 0 else 1.0

print(f"Loss globale: {loss_totale_test/len(test_loader):.4f}")
print(f"Accuracy: {metriche_test['accuracy']:.4f}")
print(f"Precision: {metriche_test['precision']:.4f}")
print(f"Recall: {metriche_test['recall']:.4f}")
print(f"Specificity: {metriche_test['specificity']:.4f}")
print(f"F1-Score: {metriche_test['f1_score']:.4f}")
print(f"AUC-ROC: {metriche_test['auc']:.4f}")
print(f"Tortuosità Media: {tortuosita_media_global:.4f}")


Metriche di Test (STARE):

Loss globale: 0.3449
Accuracy: 0.9415
Precision: 0.6404
Recall: 0.8348
Specificity: 0.9523
F1-Score: 0.7248
AUC-ROC: 0.9610
Tortuosità Media: 1.0241


### AVR

Calcolo AVR = CRAE / CRVE rispetto alla zona B. E' quindi necessario:
- distinguere vene/arterie/sfondo;
- identificare il disco ottico;
- calcolare CRAE e CRVE delle 6 arterie e delle 6 vene con calibro maggiore nella zona B (rispettivamente).

#### Segmentazione multiclasse

In RITE:
- arterie -> rosso;
- vene -> blu;
- sovrapposizione di arterie e vene -> verde;
- vasi di cui non si conosce l'identità -> bianco.

In [ ]:
# Creo una nuova classe RiteDataset. Stavolta tengo i 3 canali RGB. Non faccio Augmentation.
class RiteDataset(Dataset):
    def __init__(self, immagini_dir, av_dir, patch_size=96, patches_per_image=10, transform=None, is_train=True):

        self.immagini_dir = immagini_dir
        self.av_dir = av_dir
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.transform = transform
        self.is_train = is_train

        self.lista_immagini = sorted([f for f in os.listdir(immagini_dir) if not f.startswith('.')])
        self.lista_av = sorted([f for f in os.listdir(av_dir) if not f.startswith('.')])

    def __len__(self):
        return len(self.lista_immagini) * self.patches_per_image

    def _convert_mask_to_classes(self, mask_bgr):
        """
        Converte l'immagine BGR della maschera RITE in una matrice 2D
        contenente solo gli indici di classe (0, 1, 2, 3).
        """
        h, w, _ = mask_bgr.shape
        mask_classes = np.zeros((h, w), dtype=np.int64)

        # Rosso = [0, 0, 255]
        # Blu = [255, 0, 0]
        # Verde = [0, 255, 0]
        # Bianco = [255, 255, 255]

        is_red = (mask_bgr[:, :, 2] > 150) & (mask_bgr[:, :, 0] < 100)
        is_blue = (mask_bgr[:, :, 0] > 150) & (mask_bgr[:, :, 2] < 100)
        is_uncertain = (mask_bgr[:, :, 1] > 150)

        mask_classes[is_red] = 1        # Arteria
        mask_classes[is_blue] = 2       # Vena
        mask_classes[is_uncertain] = 3  # Sovrapposizione/incertezza (ignoro nella loss)

        return mask_classes

    def __getitem__(self,idx):
        img_idx = idx // self.patches_per_image

        img_path = os.path.join(self.immagini_dir, self.lista_immagini[img_idx])
        av_path = os.path.join(self.av_dir, self.lista_av[img_idx])

        immagine = cv2.imread(img_path)
        maschera_av_bgr = cv2.imread(av_path)

        if immagine is None or maschera_av_bgr is None:
            raise FileNotFoundError(f"Errore nel caricamento di {img_path} o {av_path}")

        # Conversione BGR --> RGB
        immagine = cv2.cvtColor(immagine, cv2.COLOR_BGR2RGB)

        # Conversione della maschera a colori in mappa di classi (0, 1, 2, 3)
        maschera_classi = self._convert_mask_to_classes(maschera_av_bgr)

        h, w, _ = immagine.shape

        max_y = h - self.patch_size
        max_x = w - self.patch_size

        if self.is_train:
            y = np.random.randint(0, max_y + 1)
            x = np.random.randint(0, max_x + 1)
        else:
            np.random.seed(idx)
            y = np.random.randint(0, max_y + 1)
            x = np.random.randint(0, max_x + 1)

        img_patch = immagine[y:y + self.patch_size, x:x + self.patch_size]
        mask_patch = maschera_classi[y:y + self.patch_size, x:x + self.patch_size]

        # Normalizzazione Min-Max
        img_patch = img_patch.astype(np.float32) / 255.0

        img_tensor = torch.from_numpy(img_patch).permute(2, 0, 1).float()

        mask_tensor = torch.from_numpy(mask_patch).long()

        return img_tensor, mask_tensor

In [ ]:
rite_training = RiteDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/rite/training/images',
    av_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/rite/training/av',
    patch_size=96,
    patches_per_image=15,
    transform=None, # no augmentation
    is_train=True
)

rite_test = RiteDataset(
    immagini_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/rite/test/images',
    av_dir='/content/drive/MyDrive/ProjectWork_CV/datasets/rite/test/av',
    patch_size=96,
    patches_per_image=10,
    transform=None,
    is_train=False
)

rite_train_loader = DataLoader(rite_training, batch_size=32, shuffle=True, num_workers=2)
rite_test_loader = DataLoader(rite_test, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# La rete ora prende 3 canali in input (anziché 1) e produce 3 canali in output (anziché 2).
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=0.2),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class RetinalUNetMulticlass(nn.Module):
    def __init__(self, in_channels=3, num_classes=3):
        super(RetinalUNetMulticlass, self).__init__()

        self.enc1 = DoubleConv(in_channels, 32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(64, 128)

        # Decoder
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(64, 32)

        # Output (Background = 0, Arteria = 1, Vena = 2)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        p1 = self.pool1(x1)

        x2 = self.enc2(p1)
        p2 = self.pool2(x2)

        b = self.bottleneck(p2)

        # Decoder
        up_b = self.up1(b)
        merge1 = torch.cat([x2, up_b], dim=1)
        d1 = self.dec1(merge1)

        up_d1 = self.up2(d1)
        merge2 = torch.cat([x1, up_d1], dim=1)
        d2 = self.dec2(merge2)

        # Output
        out = self.final_conv(d2)

        return F.log_softmax(out, dim=1)

In [ ]:
class MulticlassDiceFocalLoss(nn.Module):
    def __init__(self, num_classes=3, weight_dice=1.0, weight_focal=2.0, focal_gamma=2.0, focal_alpha=[0.1, 0.45, 0.45], ignore_index=3):
        super(MulticlassDiceFocalLoss, self).__init__()
        self.num_classes = num_classes
        self.weight_dice = weight_dice
        self.weight_focal = weight_focal
        self.focal_gamma = focal_gamma
        self.ignore_index = ignore_index
        self.register_buffer('focal_alpha', torch.tensor(focal_alpha, dtype=torch.float32))

    def forward(self, predizioni, target):

        if target.dim() == 4:
            target = target.squeeze(1)

        target = target.long()

        valid_mask = (target != self.ignore_index) # ignoro sovrapposizioni/incertezze

        target_clean = target.clone()
        target_clean[~valid_mask] = 0

        target_one_hot = F.one_hot(target_clean, num_classes=self.num_classes).permute(0, 3, 1, 2).float()

        valid_mask_expanded = valid_mask.unsqueeze(1).expand_as(target_one_hot)
        target_one_hot = target_one_hot * valid_mask_expanded

        prob = torch.exp(predizioni) * valid_mask_expanded

        # Dice Loss multiclasse
        intersezione = torch.sum(prob * target_one_hot, dim=(0, 2, 3))
        unione = torch.sum(prob, dim=(0, 2, 3)) + torch.sum(target_one_hot, dim=(0, 2, 3))
        dice = (2. * intersezione + 1e-5) / (unione + 1e-5)

        # Dice Loss media
        dice_loss = 1.0 - torch.mean(dice)

        # Focal Loss multiclasse
        ce_loss = F.nll_loss(predizioni, target_clean, reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce_loss)

        alpha_pixel = self.focal_alpha[target_clean]
        focal_loss = alpha_pixel * ((1 - pt) ** self.focal_gamma) * ce_loss

        focal_loss = focal_loss[valid_mask].mean() if valid_mask.sum() > 0 else focal_loss.mean()

        return (self.weight_dice * dice_loss) + (self.weight_focal * focal_loss)

In [ ]:
def calcola_metriche_multiclass(pred_classes, true_masks, ignore_index=3):
    """
    Calcola le metriche escludendo i pixel ignorati (ignore_index=3).
    """
    valid_mask = (true_masks != ignore_index)
    pred_valid = pred_classes[valid_mask]
    true_valid = true_masks[valid_mask]

    acc = accuracy_score(true_valid, pred_valid)
    prec = precision_score(true_valid, pred_valid, average='macro', zero_division=0)
    rec = recall_score(true_valid, pred_valid, average='macro', zero_division=0)
    f1 = f1_score(true_valid, pred_valid, average='macro', zero_division=0)

    # per arterie e vene uso anche le F1 per-classe
    f1_per_class = f1_score(true_valid, pred_valid, average=None, labels=[0, 1, 2], zero_division=0)

    return {
        "accuracy": acc, "precision": prec, "recall": rec, "f1_score": f1,
        "f1_background": f1_per_class[0],
        "f1_artery": f1_per_class[1],
        "f1_vein": f1_per_class[2]
    }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modello = RetinalUNetMulticlass(in_channels=3, num_classes=3).to(device)

checkpoint_binario_path = "/content/drive/MyDrive/ProjectWork_CV/retina_checkpoints/checkpoint_corrente.pth"
checkpoint_multiclass_dir = "/content/drive/MyDrive/ProjectWork_CV/retina_checkpoints_multiclass"
checkpoint_multiclass_path = os.path.join(checkpoint_multiclass_dir, "checkpoint_multiclass.pth")

os.makedirs(checkpoint_multiclass_dir, exist_ok=True)

if os.path.exists(checkpoint_binario_path) and not os.path.exists(checkpoint_multiclass_path):
    print("Caricamento e adattamento dei pesi dalla rete binaria...")
    checkpoint_binario = torch.load(checkpoint_binario_path, map_location=device, weights_only=False)
    state_dict_binario = checkpoint_binario['model_state_dict']

    new_state_dict = modello.state_dict()

    for name, param in state_dict_binario.items():
        if name in new_state_dict:
            if name == "enc1.block.0.weight":

                old_weight = param.data  # [32, 1, 3, 3]
                new_state_dict[name] = old_weight.repeat(1, 3, 1, 1) / 3.0
            elif name.startswith("final_conv"):
                continue
            else:
                new_state_dict[name] = param.data

    modello.load_state_dict(new_state_dict)
    print("Pesi trasferiti con successo! Il primo layer è stato espanso a 3 canali (RGB).")

criterion = MulticlassDiceFocalLoss(
    num_classes=3,
    weight_dice=1.0,
    weight_focal=2.0,
    focal_gamma=2.0,
    focal_alpha=[0.10, 0.5, 0.4],
    ignore_index=3 # ignoro sovrapposizioni/incertezze
).to(device)

optimizer = optim.Adam(
    modello.parameters(),
    lr=3e-4
    )


if 'epoca_partenza' not in locals():
    epoca_partenza = 0

storico_metriche = {
    'train_loss': [], 'train_acc': [], 'train_prec': [],
    'train_rec': [], 'train_f1': [], 'f1_artery': [], 'f1_vein': []
}

if os.path.exists(checkpoint_multiclass_path):
    print("Trovato checkpoint multiclasse! Ripristino dello stato...")
    checkpoint = torch.load(checkpoint_multiclass_path, map_location=device, weights_only=False)
    modello.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    storico_metriche = checkpoint['storico_metriche']
    epoca_partenza = checkpoint['epoca']
    print(f"Stato ripristinato. L'addestramento riprenderà dall'epoca {epoca_partenza + 1}")

Caricamento e adattamento dei pesi dalla rete binaria...
Pesi trasferiti con successo! Il primo layer è stato espanso a 3 canali (RGB).


In [ ]:
num_epoche = 50

epoca_partenza = 0

for epoca in range(epoca_partenza, num_epoche):
    print(f"Epoca {epoca+1}/{num_epoche}:")

    modello.train()
    loss_totale_train = 0

    tutti_i_target_train = []
    tutte_le_predizioni_train = []

    for immagini, maschere in rite_train_loader:
        immagini = immagini.to(device)
        maschere = maschere.to(device)

        # Forward pass
        predizioni = modello(immagini)
        loss = criterion(predizioni, maschere)

        # Backward pass e ottimizzazione
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_totale_train += loss.item()

        with torch.no_grad():
            pred_classes = torch.argmax(predizioni, dim=1)

            tutte_le_predizioni_train.append(pred_classes.cpu().numpy().ravel())
            tutti_i_target_train.append(maschere.cpu().numpy().ravel())

    tutte_le_predizioni_train = np.concatenate(tutte_le_predizioni_train)
    tutti_i_target_train = np.concatenate(tutti_i_target_train)

    metriche_train = calcola_metriche_multiclass(tutte_le_predizioni_train, tutti_i_target_train, ignore_index=3)
    loss_media_epoch = loss_totale_train / len(rite_train_loader)

    print(f"Loss: {loss_media_epoch:.4f}"
          f" Accuracy: {metriche_train['accuracy']:.4f}"
          f" Precision: {metriche_train['precision']:.4f}"
          f" Recall: {metriche_train['recall']:.4f}"
          f" F1-Macro: {metriche_train['f1_score']:.4f}"
          f" F1-Arterie: {metriche_train['f1_artery']:.4f}"
          f" F1-Vene: {metriche_train['f1_vein']:.4f}]\n"
    )

Epoca 1/50:
Loss: 0.7833 Accuracy: 0.8878 Precision: 0.5527 Recall: 0.5507 F1-Macro: 0.4861 F1-Arterie: 0.3762 F1-Vene: 0.1241]

Epoca 2/50:
Loss: 0.7136 Accuracy: 0.8972 Precision: 0.5783 Recall: 0.6171 F1-Macro: 0.5913 F1-Arterie: 0.3345 F1-Vene: 0.4775]

Epoca 3/50:
Loss: 0.6854 Accuracy: 0.8901 Precision: 0.5654 Recall: 0.6429 F1-Macro: 0.5950 F1-Arterie: 0.3270 F1-Vene: 0.5010]

Epoca 4/50:
Loss: 0.6610 Accuracy: 0.8943 Precision: 0.5826 Recall: 0.6618 F1-Macro: 0.6120 F1-Arterie: 0.3275 F1-Vene: 0.5495]

Epoca 5/50:
Loss: 0.6473 Accuracy: 0.8984 Precision: 0.5985 Recall: 0.6768 F1-Macro: 0.6278 F1-Arterie: 0.3593 F1-Vene: 0.5642]

Epoca 6/50:
Loss: 0.6322 Accuracy: 0.8981 Precision: 0.5990 Recall: 0.6793 F1-Macro: 0.6288 F1-Arterie: 0.3830 F1-Vene: 0.5434]

Epoca 7/50:
Loss: 0.6182 Accuracy: 0.9042 Precision: 0.6168 Recall: 0.6830 F1-Macro: 0.6408 F1-Arterie: 0.3878 F1-Vene: 0.5719]

Epoca 8/50:
Loss: 0.6082 Accuracy: 0.9058 Precision: 0.6199 Recall: 0.6895 F1-Macro: 0.6459 F1-Ar

In [ ]:
modello.eval()
loss_totale_test = 0

tutti_i_target_test = []
tutte_le_predizioni_test = []

with torch.no_grad():
    for immagini_test, maschere_test in rite_test_loader:
        immagini_test = immagini_test.to(device)
        maschere_test = maschere_test.to(device)

        predizioni_test = modello(immagini_test)
        loss_test = criterion(predizioni_test, maschere_test)
        loss_totale_test += loss_test.item()

        pred_classes = torch.argmax(predizioni_test, dim=1)

        tutte_le_predizioni_test.append(pred_classes.cpu().numpy().ravel())
        tutti_i_target_test.append(maschere_test.cpu().numpy().ravel())

tutte_le_predizioni_test = np.concatenate(tutte_le_predizioni_test)
tutti_i_target_test = np.concatenate(tutti_i_target_test)

metriche_test = calcola_metriche_multiclass(tutte_le_predizioni_test, tutti_i_target_test, ignore_index=3)

print("\nMetriche di Test:")
print(f"  Loss globale:     {loss_totale_test / len(rite_test_loader):.4f}")
print(f"  Accuracy:         {metriche_test['accuracy']:.4f}")
print(f"  Precision Macro:  {metriche_test['precision']:.4f}")
print(f"  Recall Macro:     {metriche_test['recall']:.4f}")
print(f"  F1-Score Macro:   {metriche_test['f1_score']:.4f}") # f1_score = (f1_background + f1_artery + f1_vein) / 3

print(f"  F1-Score Sfondo:  {metriche_test['f1_background']:.4f}")
print(f"  F1-Score Arterie: {metriche_test['f1_artery']:.4f}")
print(f"  F1-Score Vene:    {metriche_test['f1_vein']:.4f}")


Metriche di Test:
  Loss globale:     0.4278
  Accuracy:         0.9308
  Precision Macro:  0.7149
  Recall Macro:     0.7347
  F1-Score Macro:   0.7214
  F1-Score Sfondo:  0.9749
  F1-Score Arterie: 0.5413
  F1-Score Vene:    0.6480


#### Disco ottico e zona B

In [ ]:
def trova_disco_e_zona_b_stare(image, raggio_od=60):
    """
    Individua il centro del disco ottico tramite il picco di intensità sul canale rosso
    e genera la maschera ad anello per la Zona B.

    """

    # Estraggo il rosso dall'immagine
    img_bgr = cv2.imread(image)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    red_channel = img_rgb[:, :, 0]  # In RGB il Rosso è al canale 0

    # Filtro Gaussiano per sfocare i vasi e ridurre il rumore
    red_smoothed = cv2.GaussianBlur(red_channel, (51, 51), 0)

    # Cerco le coordinate del centro (pixel a intensità massima rispetto al rosso)
    _, max_val, _, max_loc = cv2.minMaxLoc(red_smoothed)
    xc, yc = max_loc

    h, w = red_channel.shape
    Y, X = np.ogrid[:h, :w]
    distanza_dal_centro = np.sqrt((X - xc)**2 + (Y - yc)**2)

    # Definizione geometrica della Zona B
    raggio_interno = 2.0 * raggio_od  # 90 pixel se raggio_od = 60
    raggio_esterno = 4.0 * raggio_od  # 120 pixel se raggio_od = 60

    # Maschera ad anello booleana
    maschera_zona_b = (distanza_dal_centro >= raggio_interno) & (distanza_dal_centro <= raggio_esterno)

    return (xc, yc), maschera_zona_b, img_rgb

#### Calcolo di CRAE e CRVE

In [ ]:
def calcola_crae(sei_max_arterie):

    """Combina i 6 calibri delle arterie per ottenere il CRAE."""

    w = sorted(sei_max_arterie) # ordinamento crescente dei calibri
    if len(w) < 2:
        return w[0] if len(w) == 1 else 0.0

    while len(w) > 1:
        wa, wb = w.pop(0), w.pop(0) # estrazione dei due rami più piccoli
        wc = 0.88 * np.sqrt(wa**2 + wb**2) # formula per la combinazione
        w.append(wc) # inserimento nella lista del nuovo valore combinato
        w = sorted(w)

    return w[0]

In [ ]:
def calcola_crve(sei_max_vene):
    """Combina i 6 calibri delle vene per ottenere il CRVE."""
    w = sorted(sei_max_vene)
    if len(w) < 2:
        return w[0] if len(w) == 1 else 0.0

    while len(w) > 1:
        wa, wb = w.pop(0), w.pop(0)
        wc = 0.95 * np.sqrt(wa**2 + wb**2)
        w.append(wc)
        w = sorted(w)

    return w[0]

In [ ]:
def estrai_sei_max_calibri(maschera_vaso_zona_b,dist_map):
    """
    Per ogni ramo vascolare della zona B viene calcolato il calibro
    sfruttando i valori della Distance Transform solo il corrispondenza
    dei pixel che costituiscono lo scheletro (asse mediano).

    """
    if np.sum(maschera_vaso_zona_b) == 0:
        return [0.0] * 6

    # riduco ogni ramo della zona B ad una linea spessa 1 pixel
    scheletro = skeletonize(maschera_vaso_zona_b)
    labeled_scheletro, num_rami = label(scheletro) # identificativo univoco per ogni sotto-ramo

    calibri_rami = []
    for i in range(1, num_rami + 1):
        pixel_ramo = (labeled_scheletro == i) # prendo solo i pixel appartenenti all'i-esimo ramo
        # Diametro = 2 * valore della Distance Transform lungo lo scheletro, solo per l'i-esimo ramo
        diametro_medio = np.mean(dist_map[pixel_ramo]) * 2.0
        calibri_rami.append(diametro_medio)

    # Ordina in modo decrescente e seleziona i primi 6
    calibri_ordinati = sorted(calibri_rami, reverse=True)
    sei_max = calibri_ordinati[:6]

    # Se ci sono meno di 6 rami, riempio con 0.0
    while len(sei_max) < 6:
        sei_max.append(0.0)

    return sei_max

In [ ]:
from PIL import Image

img_path = "/content/drive/MyDrive/ProjectWork_CV/datasets/stare/optic_disc/im0030.ppm"
raggio_od = 60

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modello.eval()


# adatto le dimensioni
def pad_to_multiple(tensor, multiple=32):
    _, _, h, w = tensor.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple

    # applica padding (destra e basso)
    tensor_padded = F.pad(tensor, (0, pad_w, 0, pad_h), mode="reflect")
    return tensor_padded, h, w


print(f"Elaborazione dell'immagine: {os.path.basename(img_path)}")

try:
    # individuazione della zona B
    (xc, yc), zona_b_mask, img_rgb = trova_disco_e_zona_b_stare(
        img_path, raggio_od=raggio_od
    )


    img_float = img_rgb.astype(np.float32)
    mean, std = img_float.mean(), img_float.std()
    img_zscore = (img_float - mean) / (std + 1e-8)

    min_val, max_val = img_zscore.min(), img_zscore.max()
    if max_val > min_val:
        img_preprocessed = (img_zscore - min_val) / (max_val - min_val)
    else:
        img_preprocessed = img_zscore - min_val

    img_tensor = (
        torch.tensor(img_preprocessed, dtype=torch.float32)
        .permute(2, 0, 1)
        .unsqueeze(0)
        .to(device)
    )


    # padding
    img_tensor_padded, orig_h, orig_w = pad_to_multiple(
        img_tensor, multiple=32
    )

    # segmentazione multiclasse
    with torch.no_grad():
        output_padded = modello(img_tensor_padded)

        output = output_padded[:, :, :orig_h, :orig_w] # rimozione padding per ottenere dimensioni originali

        pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

    maschera_arterie = pred_mask == 1
    maschera_vene = pred_mask == 2

    # mappe di distanza
    dist_map_arterie = distance_transform_edt(maschera_arterie)
    dist_map_vene = distance_transform_edt(maschera_vene)

    arterie_zona_b = maschera_arterie & zona_b_mask
    vene_zona_b = maschera_vene & zona_b_mask

    # estrazione calibri
    sei_arterie_max = estrai_sei_max_calibri(arterie_zona_b, dist_map_arterie)
    sei_vene_max = estrai_sei_max_calibri(vene_zona_b, dist_map_vene)

    # calcolo CRAE, CRVE e AVR
    crae = calcola_crae(sei_arterie_max)
    crve = calcola_crve(sei_vene_max)

    if crve > 0:
        avr = crae / crve
        print("\n--- RISULTATI ---")
        print(f"CRAE : {crae:.4f}")
        print(f"CRVE : {crve:.4f}")
        print(f"AVR  : {avr:.4f}")
    else:
        print("\nImpossibile calcolare l'AVR: CRVE pari a 0.")

except Exception as e:
    print(f"Errore durante l'elaborazione dell'immagine: {e}")

Elaborazione dell'immagine: im0030.ppm

--- RISULTATI ---
CRAE : 12.8903
CRVE : 15.4145
AVR  : 0.8362
